### Data Preparation 
#### - Pre-processing the data to make it ready for analysis

Data preprocessing - separate into parts

Data EDA 

5 borough dfs

< **TODO: Add more context, EDA, Initial processing to arrive at Borough wise DF's** >

In [2]:
import pandas as pd

### Load the datasets from the data/processed folder:

In [10]:
from pathlib import Path
import pandas as pd

# this finds the project root regardless of where the notebook is
ROOT     = Path.cwd().parent
DATA_DIR = ROOT / "data" / "processed"
#Load the datasets
manhattan_df        = pd.read_csv(DATA_DIR / "manhattan_df.csv")
queens_df       = pd.read_csv(DATA_DIR / "queens_df.csv")
bronx_df = pd.read_csv(DATA_DIR / "bronx_df.csv")
staten_island_df = pd.read_csv(DATA_DIR / "staten_df.csv")
brooklyn_df = pd.read_csv(DATA_DIR / "brooklyn_df.csv")    

In [13]:
manhattan_df.shape

(339505, 8)

In [14]:
brooklyn_df.shape

(410086, 8)

Manhattan has about 340K listings and Brooklyn has about 410K reviews in total. To proceed, we would need to create smaller samples, this makes our processing much more reasonable in terms of computational resources needed. 

### Sampling Before Borough Processing

In [25]:
listing_counts_manhattan = manhattan_df['listing_id'].value_counts() #Does not contain any language detection or analysis
listing_counts_brooklyn = brooklyn_df['listing_id'].value_counts()

high_num_review_listings_manhattan = listing_counts_manhattan[listing_counts_manhattan >= 200]
high_num_review_listings_brooklyn = listing_counts_brooklyn[listing_counts_brooklyn >= 200]

print("\nTotal number of Manhattan listings with more than 200 reviews:", len(high_num_review_listings_manhattan))
print("\nTotal number of Brooklyn listings with more than 200 reviews:", len(high_num_review_listings_brooklyn))


Total number of Manhattan listings with more than 200 reviews: 318

Total number of Brooklyn listings with more than 200 reviews: 407


If we consider listings with reviews less than 200, for a sample of around 15K reviews, the number of reviews per significant listing would average around 20. For a meaningful statistical mass for analysis, we would want the number to more, at least >30. 
So we select listings with >200 reviews, and we can sample about 36 reviews per listing for Brooklyn (15000/407) and 47 reviews per listing for Manhattan. (15000/318)


In [31]:
# ============================================================
# SAMPLING UTILITY
# ============================================================
# Default value for minimun number of reviews is set to 200, and the number of reviews per listing is set to 30. These can be adjusted as needed. 
# For Brooklyn and Manhattan we will adjust them as discussed above

def create_listing_sample(df, borough_name, min_reviews=200, reviews_per_listing=30, random_state=42):
    """
    Creates a stratified sample from listings with >= min_reviews.
    Stratified by 'rating' column within each listing.
    Falls back to random sampling if a listing lacks rating variety.
    
    Args:
        df                 : Borough DataFrame
        borough_name       : String label for logging (e.g. 'Manhattan')
        min_reviews        : Minimum reviews a listing must have to be included
        reviews_per_listing: Target number of reviews to sample per listing
        random_state       : For reproducibility
    
    Returns:
        Sampled DataFrame
    """
    
    # Step 1: Filter to listings with enough reviews
    listing_counts = df['listing_id'].value_counts()
    eligible_listings = listing_counts[listing_counts >= min_reviews].index
    df_eligible = df[df['listing_id'].isin(eligible_listings)].copy()
    
    print(f"\n{borough_name}")
    print(f"  Total listings          : {df['listing_id'].nunique():,}")
    print(f"  Eligible (>={min_reviews} reviews) : {len(eligible_listings):,}")
    print(f"  Total eligible reviews  : {len(df_eligible):,}")
    
    # Sample per listing
    df_sample = (
        df_eligible
        .groupby('listing_id', group_keys=False)
        .apply(lambda x: x.sample(
            n=min(reviews_per_listing, len(x)),  # take all if listing has fewer than target
            random_state=random_state
        ))
        .reset_index(drop=True)
    )
    
    print(f"  Final sample size       : {len(df_sample):,}")
    
    return df_sample

Note that Individual Borough df's do not have a rating available for every review, Ratings were only available at a Listing level.

In [32]:
manhattan_df.head()

,listing_id,review_id,date,reviews,neighbourhood_group,room_type,price,review_length
0,2595,17857,2009-11-21,Notre séjour de trois nuits.\r<br/>Nous avons ...,Manhattan,Entire home/apt,240.0,124
1,2595,19176,2009-12-05,Great experience.,Manhattan,Entire home/apt,240.0,2
2,2595,19760,2009-12-10,I've stayed with my friend at the Midtown Cast...,Manhattan,Entire home/apt,240.0,90
3,2595,34320,2010-04-09,"We've been staying here for about 9 nights, en...",Manhattan,Entire home/apt,240.0,66
4,2595,46312,2010-05-25,We had a wonderful stay at Jennifer's charming...,Manhattan,Entire home/apt,240.0,24


In [33]:

# ============================================================
# GENERATE SAMPLES
# ============================================================
manhattan_sample = create_listing_sample(manhattan_df, borough_name='Manhattan', min_reviews=200, reviews_per_listing=47, random_state=42)
brooklyn_sample  = create_listing_sample(brooklyn_df,  borough_name='Brooklyn', min_reviews=200, reviews_per_listing=36, random_state=42)



Manhattan
  Total listings          : 10,618
  Eligible (>=200 reviews) : 318
  Total eligible reviews  : 108,883


C:\Users\avant\AppData\Local\Temp\ipykernel_28664\3888472781.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_eligible


  Final sample size       : 14,946

Brooklyn
  Total listings          : 10,057
  Eligible (>=200 reviews) : 407
  Total eligible reviews  : 122,010
  Final sample size       : 14,652


C:\Users\avant\AppData\Local\Temp\ipykernel_28664\3888472781.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_eligible


In [35]:

# ============================================================
# SAVE
# ============================================================
ROOT     = Path.cwd().parent
SAMPLES_DIR = ROOT / "data" / "samples"
manhattan_sample.to_csv(SAMPLES_DIR / "manhattan_sample.csv", index=False)
brooklyn_sample.to_csv(SAMPLES_DIR / "brooklyn_sample.csv", index=False)

print("\nSamples saved.")


Samples saved.
